# Colab Two-Tower Model Comparison

This notebook compares trained artifacts without retraining.

It reads models from Drive:

```text
/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v1/
/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v2/
/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v3/
```

and evaluates them on the same validation candidate protocol using Gold data:

```text
/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab/
```

Silver data is not required. This notebook does not clone GitHub and does not train any model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

GOLD_ROOT = Path('/content/drive/MyDrive/recsys/data/gold/two_tower/v1_colab')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/recsys/artifacts/two_tower')
ALS_SUMMARY_PATH = Path('/content/drive/MyDrive/recsys/data/gold/als/v1_colab/evaluation_summary.json')
OUTPUT_DIR = ARTIFACT_ROOT / 'comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    'two_tower_v1': ARTIFACT_ROOT / 'baseline_v1',
    'two_tower_v2': ARTIFACT_ROOT / 'baseline_v2',
    'two_tower_v3': ARTIFACT_ROOT / 'baseline_v3',
}

print('Gold root:', GOLD_ROOT, GOLD_ROOT.exists())
print('Artifact root:', ARTIFACT_ROOT, ARTIFACT_ROOT.exists())
print('ALS summary:', ALS_SUMMARY_PATH, ALS_SUMMARY_PATH.exists())
for name, path in RUNS.items():
    print(name, path, path.exists())

In [ ]:
required_gold_paths = {
    'targets_train': GOLD_ROOT / 'targets/train',
    'targets_validation': GOLD_ROOT / 'targets/validation',
    'user_state_train': GOLD_ROOT / 'user_state/train',
    'user_state_validation': GOLD_ROOT / 'user_state/validation',
    'item_pit_train': GOLD_ROOT / 'item_features/point_in_time/train',
    'item_pit_validation': GOLD_ROOT / 'item_features/point_in_time/validation',
    'vocab_user_active_degree': GOLD_ROOT / 'vocabularies/user_active_degree',
    'vocab_video_type': GOLD_ROOT / 'vocabularies/video_type',
    'vocab_upload_type': GOLD_ROOT / 'vocabularies/upload_type',
    'numeric_stats': GOLD_ROOT / 'transforms/numeric_stats.json',
}
required_artifacts = ['config.json', 'metrics.json', 'best_model.pt']

print('Gold parquet counts:')
for name, path in required_gold_paths.items():
    if path.is_dir():
        count = len(list(path.rglob('part-*.snappy.parquet')))
        print(f'{name:28s} {count:5d} parquet files | {path}')
    else:
        print(f'{name:28s}', 'exists' if path.exists() else 'MISSING', '|', path)

print('\nRun artifacts:')
for run_name, run_dir in RUNS.items():
    for filename in required_artifacts:
        path = run_dir / filename
        print(f'{run_name:14s} {filename:18s}', 'OK' if path.exists() else 'MISSING')

In [ ]:
import importlib.util
import sys

missing = [pkg for pkg in ['torch', 'pandas', 'pyarrow', 'numpy'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    !{sys.executable} -m pip install -q {' '.join(missing)}
else:
    print('All required packages are available.')

import json
import math
import random
from dataclasses import dataclass
from typing import Mapping

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
from torch import nn
from torch.nn import functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## Feature Contract

This must match the training notebooks. Target/current-event outcomes are never used as input features.

In [ ]:
USER_CATEGORICAL = ('user_active_degree',)
ITEM_CATEGORICAL = ('video_type', 'upload_type')
USER_ID_FEATURES = ('user_id',)
ITEM_ID_FEATURES = ('video_id',)

USER_NUMERIC = (
    'is_lowactive_period', 'is_live_streamer', 'is_video_author',
    'follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days',
    *[f'onehot_feat{i}' for i in range(18)],
    'user_hist_events', 'user_hist_long_view_rate', 'user_hist_like_rate',
    'user_hist_comment_rate', 'user_hist_forward_rate', 'user_hist_follow_rate',
    'user_hist_hate_rate', 'user_hist_avg_watch_ratio', 'user_hist_avg_play_time_sec',
    'session_event_index', 'session_elapsed_sec', 'session_prior_long_view_rate',
    'session_prior_like_rate', 'session_prior_hate_rate', 'session_prior_avg_watch_ratio',
    'session_vs_user_long_view_delta', 'session_vs_user_watch_ratio_delta',
    'event_hour', 'event_dayofweek', 'tab', 'is_rand',
)
ITEM_NUMERIC = (
    'video_duration_sec', 'aspect_ratio', 'visible_status', 'music_type',
    'upload_age_days_at_event', 'item_hist_events', 'item_hist_long_view_rate',
    'item_hist_like_rate', 'item_hist_hate_rate', 'item_hist_avg_watch_ratio',
)
EMBED_DIMS = {
    'user_id': 8,
    'video_id': 16,
    'user_active_degree': 4,
    'video_type': 3,
    'upload_type': 8,
}
print('user numeric:', len(USER_NUMERIC))
print('item numeric:', len(ITEM_NUMERIC))

In [ ]:
@dataclass
class Vocabulary:
    feature_name: str
    value_to_index: dict
    oov_index: int = 0

    @property
    def size(self):
        return max(self.value_to_index.values(), default=0) + 1

    def encode_many(self, values):
        return [self.value_to_index.get(v, self.oov_index) if pd.notna(v) else self.oov_index for v in values]

    @classmethod
    def from_values(cls, feature_name, values):
        unique = sorted({v for v in values if pd.notna(v)})
        return cls(feature_name, {v: i + 1 for i, v in enumerate(unique)})


def parquet_files(path: Path):
    path = Path(path)
    if path.is_file():
        return [path] if path.name.endswith('.parquet') else []
    if not path.exists():
        raise FileNotFoundError(f'Missing path: {path}')
    files = sorted(p for p in path.rglob('*.parquet') if p.is_file() and not p.name.startswith('.'))
    if not files:
        visible = sorted(x.name for x in path.iterdir())[:30] if path.is_dir() else []
        raise FileNotFoundError(f'No parquet files found under {path}. Visible entries: {visible}')
    return files


def read_parquet_head(path: Path, max_rows: int | None, columns=None) -> pd.DataFrame:
    files = parquet_files(path)
    dataset = ds.dataset([str(f) for f in files], format='parquet')
    if columns is not None:
        missing = sorted(set(columns) - set(dataset.schema.names))
        if missing:
            raise ValueError(f'{path} is missing columns {missing}. Available columns: {dataset.schema.names}')
    if max_rows is None:
        return dataset.to_table(columns=columns).to_pandas()
    return dataset.head(max_rows, columns=columns).to_pandas()


def load_parquet_vocabulary(path: Path, feature_name: str) -> Vocabulary:
    df = read_parquet_head(path, None)
    return Vocabulary(feature_name, dict(zip(df[feature_name].tolist(), df[f'{feature_name}_idx'].astype(int).tolist())))


class NumericalPreprocessor:
    def __init__(self, stats: Mapping[str, Mapping[str, float]], clip=True, normalize=True, eps=1e-6):
        self.stats = stats
        self.clip = clip
        self.normalize = normalize
        self.eps = eps

    @classmethod
    def from_json(cls, path: Path):
        payload = json.loads(path.read_text())
        return cls(payload.get('features', payload))

    def transform_tensor(self, frame: pd.DataFrame, features: tuple[str, ...]) -> torch.Tensor:
        cols = []
        for feat in features:
            st = self.stats[feat]
            values = pd.to_numeric(frame[feat], errors='coerce').astype('float32')
            values = values.fillna(float(st.get('mean', 0.0) or 0.0))
            if self.clip:
                values = values.clip(float(st.get('p01', st.get('min', -np.inf))), float(st.get('p99', st.get('max', np.inf))))
            if self.normalize:
                mean = float(st.get('mean', 0.0) or 0.0)
                std = float(st.get('stddev', 0.0) or 0.0)
                values = (values - mean) / std if std > self.eps else values * 0.0
            cols.append(np.nan_to_num(values.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0))
        return torch.tensor(np.stack(cols, axis=1), dtype=torch.float32)


class BatchPreprocessor:
    def __init__(self, vocabularies, numerical):
        self.vocabularies = vocabularies
        self.numerical = numerical

    def encode_categorical(self, frame, features):
        return {f: torch.tensor(self.vocabularies[f].encode_many(frame[f].tolist()), dtype=torch.long) for f in features}

    def encode_user(self, frame):
        return self.encode_categorical(frame, USER_ID_FEATURES + USER_CATEGORICAL), self.numerical.transform_tensor(frame, USER_NUMERIC)

    def encode_item(self, frame):
        return self.encode_categorical(frame, ITEM_ID_FEATURES + ITEM_CATEGORICAL), self.numerical.transform_tensor(frame, ITEM_NUMERIC)

In [ ]:
def load_joined_split(split: str, max_examples: int | None) -> pd.DataFrame:
    target_cols = [
        'example_id', 'user_id', 'video_id', 'as_of_time',
        'target_class', 'is_positive', 'is_observed_negative', 'is_ambiguous',
    ]
    user_cols = ['example_id', *USER_ID_FEATURES, *USER_CATEGORICAL, *USER_NUMERIC]
    item_cols = ['example_id', *ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC]

    targets = read_parquet_head(GOLD_ROOT / 'targets' / split, max_examples, target_cols)
    user_state = read_parquet_head(GOLD_ROOT / 'user_state' / split, max_examples, user_cols)
    item_features = read_parquet_head(GOLD_ROOT / 'item_features' / 'point_in_time' / split, max_examples, item_cols)

    frame = (
        targets.merge(user_state, on=['example_id', 'user_id'], how='inner')
        .merge(item_features, on=['example_id', 'video_id'], how='inner')
    )
    frame = frame[frame['target_class'].eq('STRONG_POSITIVE')]
    return frame.reset_index(drop=True)


def load_item_feature_pool(split: str, max_examples: int | None = None) -> pd.DataFrame:
    item_cols = [*ITEM_ID_FEATURES, *ITEM_CATEGORICAL, *ITEM_NUMERIC]
    return read_parquet_head(
        GOLD_ROOT / 'item_features' / 'point_in_time' / split,
        max_examples,
        item_cols,
    ).drop_duplicates('video_id', keep='last').reset_index(drop=True)

print('Loading shared validation frame...')
validation_frame = load_joined_split('validation', None)
print('validation positives:', validation_frame.shape)

print('Loading shared validation candidate items...')
candidate_items = load_item_feature_pool('validation', None)
print('candidate items:', candidate_items.shape)

In [ ]:
class FeatureEncoder(nn.Module):
    def __init__(self, table_sizes: dict, numeric_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleDict({
            name: nn.Embedding(size, EMBED_DIMS[name], padding_idx=0)
            for name, size in table_sizes.items()
        })
        self.numeric_dim = numeric_dim
        self.output_dim = sum(EMBED_DIMS[name] for name in table_sizes) + numeric_dim

    def forward(self, categorical, numeric):
        pieces = [self.embeddings[name](categorical[name].long()) for name in self.embeddings]
        pieces.append(numeric.float())
        return torch.cat(pieces, dim=1)


def build_mlp(input_dim, hidden_dims, output_dim, dropout):
    layers = []
    prev = input_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class TwoTowerModel(nn.Module):
    def __init__(self, config, vocabularies):
        super().__init__()
        self.config = config
        self.user_encoder = FeatureEncoder(
            {'user_id': vocabularies['user_id'].size, 'user_active_degree': vocabularies['user_active_degree'].size},
            len(USER_NUMERIC),
        )
        self.item_encoder = FeatureEncoder(
            {'video_id': vocabularies['video_id'].size, 'video_type': vocabularies['video_type'].size, 'upload_type': vocabularies['upload_type'].size},
            len(ITEM_NUMERIC),
        )
        self.user_tower = build_mlp(self.user_encoder.output_dim, config['user_hidden_dims'], config['retrieval_dim'], config['dropout'])
        self.item_tower = build_mlp(self.item_encoder.output_dim, config['item_hidden_dims'], config['retrieval_dim'], config['dropout'])

    def encode_user(self, user_cat, user_num):
        return F.normalize(self.user_tower(self.user_encoder(user_cat, user_num)), dim=1)

    def encode_item(self, item_cat, item_num):
        return F.normalize(self.item_tower(self.item_encoder(item_cat, item_num)), dim=1)


def load_run_config(run_dir: Path):
    config_path = run_dir / 'config.json'
    model_path = run_dir / 'best_model.pt'
    config = json.loads(config_path.read_text()) if config_path.exists() else None
    checkpoint = torch.load(model_path, map_location='cpu')
    if config is None:
        config = checkpoint.get('config')
    if config is None:
        raise ValueError(f'No config found for {run_dir}')
    return config, checkpoint


def build_vocabularies_for_run(config):
    max_train = config.get('max_train_examples')
    train_frame = load_joined_split('train', max_train)
    if config.get('full_item_vocab_from_train_items'):
        item_pool = load_item_feature_pool('train', config.get('max_negative_item_examples'))
        video_values = item_pool['video_id'].tolist()
    else:
        video_values = train_frame['video_id'].tolist()
    vocabularies = {
        'user_active_degree': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/user_active_degree', 'user_active_degree'),
        'video_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/video_type', 'video_type'),
        'upload_type': load_parquet_vocabulary(GOLD_ROOT / 'vocabularies/upload_type', 'upload_type'),
        'user_id': Vocabulary.from_values('user_id', train_frame['user_id'].tolist()),
        'video_id': Vocabulary.from_values('video_id', video_values),
    }
    return vocabularies, train_frame.shape[0]


numerical = NumericalPreprocessor.from_json(GOLD_ROOT / 'transforms/numeric_stats.json')

In [ ]:
def encode_user_frame(model, preprocessor, frame: pd.DataFrame, batch_size: int):
    outputs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(frame), batch_size):
            chunk = frame.iloc[start:start + batch_size]
            user_cat, user_num = preprocessor.encode_user(chunk)
            user_cat = {k: v.to(device) for k, v in user_cat.items()}
            user_num = user_num.to(device)
            outputs.append(model.encode_user(user_cat, user_num).cpu())
    return torch.cat(outputs, dim=0) if outputs else torch.empty((0, model.config['retrieval_dim']))


def encode_item_frame(model, preprocessor, frame: pd.DataFrame, batch_size: int):
    outputs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(frame), batch_size):
            chunk = frame.iloc[start:start + batch_size]
            item_cat, item_num = preprocessor.encode_item(chunk)
            item_cat = {k: v.to(device) for k, v in item_cat.items()}
            item_num = item_num.to(device)
            outputs.append(model.encode_item(item_cat, item_num).cpu())
            if (start // batch_size + 1) % 10 == 0:
                print(f'encoded item rows: {start + len(chunk):,}/{len(frame):,}')
    return torch.cat(outputs, dim=0) if outputs else torch.empty((0, model.config['retrieval_dim']))


def merge_topk(previous_scores, previous_indices, new_scores, new_indices, k):
    scores = torch.cat([previous_scores, new_scores], dim=1)
    indices = torch.cat([previous_indices, new_indices], dim=1)
    top_scores, positions = torch.topk(scores, k=min(k, scores.shape[1]), dim=1)
    top_indices = torch.gather(indices, 1, positions)
    return top_scores, top_indices


def candidate_topk_indices(user_embeddings, item_embeddings, k, user_batch_size, item_batch_size):
    all_top_indices = []
    item_count = item_embeddings.shape[0]
    for user_start in range(0, user_embeddings.shape[0], user_batch_size):
        user_chunk = user_embeddings[user_start:user_start + user_batch_size].to(device)
        current_k = min(k, item_count)
        top_scores = torch.full((user_chunk.shape[0], 0), -float('inf'), device=device)
        top_indices = torch.empty((user_chunk.shape[0], 0), dtype=torch.long, device=device)
        for item_start in range(0, item_count, item_batch_size):
            item_chunk = item_embeddings[item_start:item_start + item_batch_size].to(device)
            scores = user_chunk @ item_chunk.T
            chunk_k = min(current_k, scores.shape[1])
            chunk_scores, chunk_local_indices = torch.topk(scores, k=chunk_k, dim=1)
            chunk_global_indices = chunk_local_indices + item_start
            top_scores, top_indices = merge_topk(top_scores, top_indices, chunk_scores, chunk_global_indices, current_k)
        all_top_indices.append(top_indices.cpu())
        print(f'scored users: {min(user_start + user_chunk.shape[0], user_embeddings.shape[0]):,}/{user_embeddings.shape[0]:,}')
    return torch.cat(all_top_indices, dim=0)


def ranking_metrics_from_topk(query_user_ids, candidate_video_ids, top_indices, relevance_by_user, top_ks):
    max_k = top_indices.shape[1]
    metrics = {}
    evaluated = 0
    accum = {f'hitrate@{k}': 0.0 for k in top_ks}
    accum.update({f'recall@{k}': 0.0 for k in top_ks})
    accum.update({f'ndcg@{k}': 0.0 for k in top_ks})

    for row_idx, user_id in enumerate(query_user_ids):
        relevant = relevance_by_user.get(user_id, set())
        if not relevant:
            continue
        evaluated += 1
        ranked_items = [int(candidate_video_ids[i]) for i in top_indices[row_idx, :max_k].tolist()]
        for k in top_ks:
            prefix = ranked_items[:min(k, len(ranked_items))]
            hits = [1 if item in relevant else 0 for item in prefix]
            hit_count = sum(hits)
            accum[f'hitrate@{k}'] += 1.0 if hit_count > 0 else 0.0
            accum[f'recall@{k}'] += hit_count / len(relevant)
            dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
            ideal_hits = min(len(relevant), k)
            idcg = sum(1.0 / np.log2(rank + 2) for rank in range(ideal_hits))
            accum[f'ndcg@{k}'] += dcg / idcg if idcg > 0 else 0.0

    if evaluated == 0:
        return {'evaluated_users': 0}
    return {key: value / evaluated for key, value in accum.items()} | {'evaluated_users': evaluated, 'candidate_items': len(candidate_video_ids)}

## Reported Metrics From Saved Artifacts

This table simply reads `metrics.json` from each run. It does not recompute anything.

In [ ]:
reported_rows = []
for run_name, run_dir in RUNS.items():
    metrics_path = run_dir / 'metrics.json'
    if not metrics_path.exists():
        reported_rows.append({'model': run_name, 'status': 'missing metrics.json'})
        continue
    metrics = json.loads(metrics_path.read_text())
    candidate = metrics.get('candidate_validation') or {}
    in_batch = metrics.get('final_validation') or {}
    reported_rows.append({
        'model': run_name,
        'status': 'ok',
        'best_epoch': metrics.get('best_epoch'),
        'candidate_hitrate@10': candidate.get('hitrate@10'),
        'candidate_ndcg@10': candidate.get('ndcg@10'),
        'candidate_recall@10': candidate.get('recall@10'),
        'candidate_hitrate@50': candidate.get('hitrate@50'),
        'candidate_ndcg@50': candidate.get('ndcg@50'),
        'in_batch_ndcg@50': in_batch.get('ndcg@50'),
    })
reported_df = pd.DataFrame(reported_rows)
display(reported_df)

## Shared Candidate Evaluation

This recomputes V1/V2/V3 on the same validation users, same candidate items, same relevance labels, and same `K` values. It can take a few minutes because it scores each model against ~850k validation candidate items.

In [ ]:
TOP_KS = [10, 50, 100]
USER_BATCH_SIZE = 64
ITEM_BATCH_SIZE = 65536
MAX_USERS = None      # set to e.g. 500 for faster debug
MAX_ITEMS = None      # set to e.g. 300_000 for faster debug

relevance_by_user = validation_frame.groupby('user_id')['video_id'].apply(lambda s: set(map(int, s))).to_dict()
query_users = (
    validation_frame.sort_values('as_of_time')
    .drop_duplicates('user_id', keep='last')
    .reset_index(drop=True)
)
if MAX_USERS is not None:
    query_users = query_users.head(MAX_USERS)

candidate_eval_items = candidate_items.copy()
if MAX_ITEMS is not None and len(candidate_eval_items) > MAX_ITEMS:
    positive_items = set().union(*relevance_by_user.values()) if relevance_by_user else set()
    positive_candidates = candidate_eval_items[candidate_eval_items['video_id'].isin(positive_items)]
    remaining_n = max(MAX_ITEMS - len(positive_candidates), 0)
    other_candidates = candidate_eval_items[~candidate_eval_items['video_id'].isin(positive_items)].sample(
        n=min(remaining_n, max(len(candidate_eval_items) - len(positive_candidates), 0)),
        random_state=42,
    )
    candidate_eval_items = pd.concat([positive_candidates, other_candidates], ignore_index=True).drop_duplicates('video_id')

print('query users:', len(query_users))
print('candidate items:', len(candidate_eval_items))

In [ ]:
shared_rows = []
for run_name, run_dir in RUNS.items():
    print('\n===', run_name, '===')
    config, checkpoint = load_run_config(run_dir)
    vocabularies, train_positive_rows = build_vocabularies_for_run(config)
    preprocessor = BatchPreprocessor(vocabularies, numerical)
    model = TwoTowerModel(config, vocabularies).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print('train_positive_rows used for vocab:', train_positive_rows)
    print('vocab sizes:', {k: v.size for k, v in vocabularies.items()})
    user_embeddings = encode_user_frame(model, preprocessor, query_users, USER_BATCH_SIZE)
    item_embeddings = encode_item_frame(model, preprocessor, candidate_eval_items, ITEM_BATCH_SIZE)
    top_indices = candidate_topk_indices(user_embeddings, item_embeddings, max(TOP_KS), USER_BATCH_SIZE, ITEM_BATCH_SIZE)
    metrics = ranking_metrics_from_topk(
        query_users['user_id'].astype(int).tolist(),
        candidate_eval_items['video_id'].astype(int).to_numpy(),
        top_indices,
        relevance_by_user,
        TOP_KS,
    )
    row = {'model': run_name, **metrics}
    shared_rows.append(row)
    print(row)

shared_df = pd.DataFrame(shared_rows)
display(shared_df)

## Add ALS / Popularity Reference

These are the saved Spark ALS and popularity metrics. They may still have a different candidate universe, so use the shared Two-Tower rows above for direct V1/V2/V3 comparison, and treat ALS/Popularity as reference until ALS is re-scored on the same candidate set.

In [ ]:
def first_metric_block(payload, keys):
    for key in keys:
        if isinstance(payload.get(key), dict):
            return payload[key]
    nested = payload.get('baseline_summary') or {}
    for key in keys:
        if isinstance(nested.get(key), dict):
            return nested[key]
    return {}


reference_rows = []
if ALS_SUMMARY_PATH.exists():
    als = json.loads(ALS_SUMMARY_PATH.read_text())
    print('ALS summary top-level keys:', sorted(als.keys()))
    if isinstance(als.get('baseline_summary'), dict):
        print('ALS baseline_summary keys:', sorted(als['baseline_summary'].keys()))

    reference_specs = [
        ('popularity_saved', ['popularity_test', 'final_popularity_test']),
        ('als_saved', ['als_test', 'final_als_test']),
    ]
    for model_name, keys in reference_specs:
        values = first_metric_block(als, keys)
        if not values:
            print(f'WARNING: no metrics found for {model_name}; tried keys={keys}')
        reference_rows.append({
            'model': model_name,
            'hitrate@10': values.get('hitrate@10'),
            'recall@10': values.get('recall@10'),
            'ndcg@10': values.get('ndcg@10'),
            'evaluated_users': values.get('evaluated_users'),
            'candidate_items': None,
        })
else:
    print('ALS summary not found:', ALS_SUMMARY_PATH)
reference_df = pd.DataFrame(reference_rows)
combined_df = pd.concat([reference_df, shared_df], ignore_index=True, sort=False)
display(combined_df)

reported_df.to_csv(OUTPUT_DIR / 'reported_metrics.csv', index=False)
shared_df.to_csv(OUTPUT_DIR / 'shared_candidate_metrics.csv', index=False)
combined_df.to_csv(OUTPUT_DIR / 'combined_comparison_metrics.csv', index=False)
print('Saved comparison CSVs to:', OUTPUT_DIR)

In [ ]:
# Optional quick bar charts.
import matplotlib.pyplot as plt

plot_df = combined_df.dropna(subset=['hitrate@10', 'ndcg@10']).copy()
for metric in ['hitrate@10', 'ndcg@10']:
    ax = plot_df.plot(kind='bar', x='model', y=metric, legend=False, figsize=(8, 4), title=metric)
    ax.set_ylabel(metric)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()